In [ ]:
# Description of the Model or Regression Problem

### Astronomical Regression Problem: Predicting Stellar Surface Gravity from Effective Temperature

In this assignment, we design a simple feedforward neural network for a regression task in astrophysics. The problem is to predict the logarithmic surface gravity (log g) of a star based on its effective temperature (Teff). Surface gravity is a key parameter in stellar astrophysics, influencing spectral line broadening and atmospheric models. Effective temperature is derived from photometry or spectroscopy.

For this version, we use a subset of the APOGEE DR17 dataset (from SDSS), which provides spectroscopic parameters for ~430,000 stars. We load ~10,000 samples from the FITS file for training to handle large files efficiently:  
- Input (x): Teff (normalized).  
- Target (y): log g.  

This is a real astrophysical regression problem, earning bonus points. 
The network has exactly 5 neurons: 1 input, 3 hidden (ReLU), 1 output (linear). Loss: MSE. We demonstrate gradient descent behavior with LR=0.1 (monotonic decrease) and LR=1.0 (possible increases).

In [ ]:
# Load and Preprocess APOGEE DR17 FITS Dataset (Optimized for Large Files)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from astropy.table import Table  # Better for FITS handling

# Path to your FITS file
fits_path = '/home/jfir3/Downloads/allStar-dr17-synspec_rev1.fits'  # Update if different

# Read the FITS file using astropy Table (handles endianness and structure better)
table = Table.read(fits_path, hdu=1)  # HDU 1 is the data table

# Convert to pandas DataFrame (select scalar columns to avoid multidimensional issues)
scalar_columns = ['TEFF', 'LOGG', 'ASPCAPFLAG']
df = table[scalar_columns].to_pandas()[:1000]  # Load first 1,000 rows only
df.info()

# Filter for quality (same as before)
df = df[(df['ASPCAPFLAG'] == 0) & 
        (df['TEFF'] > 3000) & (df['TEFF'] < 8000) & 
        (df['LOGG'] > 0) & (df['LOGG'] < 6)]
df = df.dropna(subset=['TEFF', 'LOGG'])

# Prepare data
X = df['TEFF'].values.reshape(-1, 1)
y = df['LOGG'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).flatten()

print(f"Loaded {len(X)} samples from FITS file (scalar columns only).")
print(f"Teff range: {X.min():.0f} - {X.max():.0f} K")
print(f"log g range: {y.min():.2f} - {y.max():.2f}")
print(f"Sample data:\n{df[['TEFF', 'LOGG']].head()}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")


In [ ]:
# Network Structure and Graph Illustration
import matplotlib.pyplot as plt

print("Network Structure (ASCII Diagram):")
print("""
Input (x: Teff) 
    |
    | w11, b1 --> ReLU --> h1
    | w12, b2 --> ReLU --> h2
    

In [ ]:

#### Cell 4: Markdown - Equations
Forward Pass Equations
Let the input be $ x $ (Teff, normalized).

Hidden pre-activations:
$ z_1 = w_{11} x + b_1 $
$ z_2 = w_{12} x + b_2 $
$ z_3 = w_{13} x + b_3 $

Hidden activations (ReLU):
$ h_1 = \max(0, z_1) $
$ h_2 = \max(0, z_2) $
$ h_3 = \max(0, z_3) $

Output pre-activation (linear):
$ \hat{y} = w_{o1} h_1 + w_{o2} h_2 + w_{o3} h_3 + b_o $

Loss:
$ L = \frac{1}{2} (\hat{y} - y)^2 $

Backward Pass: Gradients (Backpropagation)
Let $ \delta_o = \frac{\partial L}{\partial \hat{y}} = (\hat{y} - y) $.

Output Layer Gradients
$ \frac{\partial L}{\partial w_{o1}} = \delta_o \cdot h_1 $
$ \frac{\partial L}{\partial w_{o2}} = \delta_o \cdot h_2 $
$ \frac{\partial L}{\partial w_{o3}} = \delta_o \cdot h_3 $
$ \frac{\partial L}{\partial b_o} = \delta_o $

Hidden Layer Gradients (ReLU derivative: 1 if $ z_i > 0 $, else 0)
Hidden errors:
$ \delta_1 = \delta_o \cdot w_{o1} \cdot \mathbb{I}(z_1 > 0) $
$ \delta_2 = \delta_o \cdot w_{o2} \cdot \mathbb{I}(z_2 > 0) $
$ \delta_3 = \delta_o \cdot w_{o3} \cdot \mathbb{I}(z_3 > 0) $

Input-to-hidden gradients:
$ \frac{\partial L}{\partial w_{11}} = \delta_1 \cdot x $
$ \frac{\partial L}{\partial b_1} = \delta_1 $
$ \frac{\partial L}{\partial w_{12}} = \delta_2 \cdot x $
$ \frac{\partial L}{\partial b_2} = \delta_2 $
$ \frac{\partial L}{\partial w_{13}} = \delta_3 \cdot x $
$ \frac{\partial L}{\partial b_3} = \delta_3 $

These are averaged over the batch in the code.


In [ ]:

#### Cell 5: Code - Modified SimpleNN for Batch Training
```python
# Modified SimpleNN for Batch Training with APOGEE Data
class SimpleNNBatch:
    def __init__(self, X, y, init_weights, batch_size=None):
        self.X = np.array(X)
        self.y = np.array(y)
        self.n_samples = len(X)
        if batch_size is None:
            batch_size = self.n_samples
        self.batch_size = batch_size
        self.weights = np.array(init_weights, dtype=float)
        self.history = []
    
    def forward(self, X_batch):
        w11, b1, w12, b2, w13, b3, wo1, wo2, wo3, bo = self.weights
        z1 = w11 * X_batch + b1
        h1 = np.maximum(0, z1)
        z2 = w12 * X_batch + b2
        h2 = np.maximum(0, z2)
        z3 = w13 * X_batch + b3
        h3 = np.maximum(0, z3)
        y_pred = wo1 * h1 + wo2 * h2 + wo3 * h3 + bo
        return z1, h1, z2, h2, z3, h3, y_pred
    
    def backward(self, X_batch, y_batch, y_pred_batch):
        w11, b1, w12, b2, w13, b3, wo1, wo2, wo3, bo = self.weights
        z1, h1, z2, h2, z3, h3, _ = self.forward(X_batch)
        delta_o = y_pred_batch - y_batch
        loss = 0.5 * np.mean(delta_o ** 2)
        grad_wo1 = np.mean(delta_o * h1)
        grad_wo2 = np.mean(delta_o * h2)
        grad_wo3 = np.mean(delta_o * h3)
        grad_bo = np.mean(delta_o)
        I1 = (z1 > 0).astype(float)
        I2 = (z2 > 0).astype(float)
        I3 = (z3 > 0).astype(float)
        delta1 = delta_o * wo1 * I1
        delta2 = delta_o * wo2 * I2
        delta3 = delta_o * wo3 * I3
        grad_w11 = np.mean(delta1 * X_batch)
        grad_b1 = np.mean(delta1)
        grad_w12 = np.mean(delta2 * X_batch)
        grad_b2 = np.mean(delta2)
        grad_w13 = np.mean(delta3 * X_batch)
        grad_b3 = np.mean(delta3)
        grads = np.array([grad_w11, grad_b1, grad_w12, grad_b2, grad_w13, grad_b3,
                          grad_wo1, grad_wo2, grad_wo3, grad_bo])
        mean_outputs = [np.mean(z1), np.mean(h1), np.mean(z2), np.mean(h2), 
                        np.mean(z3), np.mean(h3), np.mean(y_pred_batch)]
        return grads, loss, mean_outputs
    
    def step(self, eta, epoch=0):
        indices = np.random.choice(self.n_samples, self.batch_size, replace=False)
        X_batch = self.X[indices]
        y_batch = self.y[indices]
        z1, h1, z2, h2, z3, h3, y_pred_batch = self.forward(X_batch)
        grads, loss, mean_outputs = self.backward(X_batch, y_batch, y_pred_batch)
        self.weights -= eta * grads
        step_info = {
            'epoch': epoch,
            'weights': self.weights.copy(),
            'grads': grads,
            'outputs': mean_outputs,
            'loss': loss
        }
        self.history.append(step_info)
        return loss

# Initialize and train
init_weights = [0.5, -0.2, 0.3, 0.1, -0.4, 0.2, 1.0, 0.5, -0.5, 2.0]

def run_training_batch(X_train, y_train, eta, steps=10, batch_size=100):
    nn = SimpleNNBatch(X_train, y_train, init_weights, batch_size=batch_size)
    losses = []
    for step in range(steps):
        loss = nn.step(eta, epoch=step)
        losses.append(loss)
        print(f"Step {step+1}, Loss: {loss:.6f}")
    return nn, losses

# Run for LR=0.1
nn1, losses1 = run_training_batch(X_train, y_train, 0.1, steps=10, batch_size=min(100, len(X_train)))
print("Losses for LR=0.1:", losses1)
print("Monotonic decrease:", all(losses1[i] >= losses1[i+1] for i in range(len(losses1)-1)))

# Run for LR=1.0
nn2, losses2 = run_training_batch(X_train, y_train, 1.0, steps=10, batch_size=min(100, len(X_train)))
print("Losses for LR=1.0:", losses2)
print("Has increases:", any(losses2[i] < losses2[i+1] for i in range(len(losses2)-1)))

In [ ]:
# Generate Tables for LR=0.1
def create_table_batch(nn, lr):
    data = []
    for i, step_info in enumerate(nn.history):
        w = step_info['weights']
        g = step_info['grads']
        outs = step_info['outputs']
        l = step_info['loss']
        row = {
            'Step': i+1,
            'w11': f"{w[0]:.4f}", 'b1': f"{w[1]:.4f}", 'w12': f"{w[2]:.4f}", 'b2': f"{w[3]:.4f}",
            'w13': f"{w[4]:.4f}", 'b3': f"{w[5]:.4f}", 'wo1': f"{w[6]:.4f}", 'wo2': f"{w[7]:.4f}",
            'wo3': f"{w[8]:.4f}", 'bo': f"{w[9]:.4f}",
            'g_w11': f"{g[0]:.4f}", 'g_b1': f"{g[1]:.4f}", 'g_w12': f"{g[2]:.4f}", 'g_b2': f"{g[3]:.4f}",
            'g_w13': f"{g[4]:.4f}", 'g_b3': f"{g[5]:.4f}", 'g_wo1': f"{g[6]:.4f}", 'g_wo